# 03 — Analisis Kesalahan Model

**Tujuan:** membaca false negative dan false positive pada validation, mencari pola, dan menghubungkannya dengan keterbatasan data.

**Prasyarat:** `models/indobert_real/val_predictions.csv` atau `models/indobert_aug/val_predictions.csv` dari notebook `06` atau training lokal.

**Estimasi waktu:** 2 menit.

## Glosarium

- **False negative (FN)**: bully yang diprediksi non-bully.
- **False positive (FP)**: non-bully yang diprediksi bully.
- **Subgroup**: kelompok berdasarkan label tambahan `HS`/`Abusive`.
- **Threshold**: ambang probabilitas; mengubahnya menggeser FN/FP.

## 1. Muat prediksi validation

Threshold diambil dari `out/model_a/threshold.json` atau `out/model_b/threshold.json` sesuai `MODEL_KEY`; jika belum ada, 0,5.
Prediksi dihitung ulang dari probabilitas supaya konsisten. Notebook ini tidak
menyentuh test dan tidak mengubah model.

In [ ]:
import json

import pandas as pd

from cyberbully.config import LABEL_COLUMN, MODELS_DIR, OUT_DIR, TEXT_COLUMN

MODEL_KEY = "A"  # ganti "B" untuk model augmentasi
MODEL_DIRS = {"A": MODELS_DIR / "indobert_real", "B": MODELS_DIR / "indobert_aug"}
MODEL_LABELS = {"A": "real-only", "B": "real + sintetis"}
model_dir = MODEL_DIRS[MODEL_KEY]
pred_path = model_dir / "val_predictions.csv"
threshold_path = OUT_DIR / f"model_{MODEL_KEY.lower()}" / "threshold.json"

df = pd.read_csv(pred_path) if pred_path.exists() else None
threshold = 0.5
if df is not None and threshold_path.exists():
    threshold = float(json.loads(threshold_path.read_text())["threshold"])
if df is not None:
    df["prediction"] = (df["probability"] >= threshold).astype(int)
    print("model:", MODEL_KEY, MODEL_LABELS[MODEL_KEY])
    print("baris:", len(df), "| threshold:", threshold)
else:
    print("Prediksi belum ada:", pred_path)


## 2. Contoh kesalahan

Kita menampilkan beberapa FN dan FP agar polanya terlihat. Teks hanya muncul di
runtime notebook; output tidak boleh di-commit karena berisi teks pengguna.
Perhatikan apakah kesalahan terjadi pada sindiran, kata kasar akrab, atau teks
yang memang ambigu.

In [ ]:
if df is not None:
    fn = df[(df[LABEL_COLUMN] == 1) & (df["prediction"] == 0)]
    fp = df[(df[LABEL_COLUMN] == 0) & (df["prediction"] == 1)]
    print("false negative:", len(fn), "| false positive:", len(fp))
    print("\nContoh FN:")
    for text, prob in fn[[TEXT_COLUMN, "probability"]].head(5).itertuples(index=False):
        print(f"- p={prob:.2f} | {text}")
    print("\nContoh FP:")
    for text, prob in fp[[TEXT_COLUMN, "probability"]].head(5).itertuples(index=False):
        print(f"- p={prob:.2f} | {text}")
else:
    print("Dilewati: prediksi belum tersedia.")

## 3. Pola per subgroup

Label `HS` dan `Abusive` menunjukkan jenis konten. Kita membandingkan jumlah FN
dan FP per subgroup. Angka per subgroup bisa kecil, jadi jangan menyimpulkan pola
dari belasan contoh; gunakan sebagai petunjuk untuk inspeksi manual.

In [ ]:
if df is not None and {"HS", "Abusive"}.issubset(df.columns):
    df["subgroup"] = "HS=" + df["HS"].astype(str) + ", Abusive=" + df["Abusive"].astype(str)
    summary = (
        df.assign(
            fn=(df[LABEL_COLUMN] == 1) & (df["prediction"] == 0),
            fp=(df[LABEL_COLUMN] == 0) & (df["prediction"] == 1),
        )
        .groupby("subgroup")
        .agg(jumlah=(TEXT_COLUMN, "size"), fn=("fn", "sum"), fp=("fp", "sum"))
        .sort_values("jumlah", ascending=False)
    )
    print(summary.to_string())
else:
    print("Kolom subgroup tidak tersedia atau prediksi belum ada.")

## 4. Sebaran probabilitas

Histogram memperlihatkan seberapa yakin model. Banyak probabilitas di sekitar
threshold berarti model ragu; ekor di sisi salah menunjukkan error yang
sistematis. Ini membantu memilih apakah threshold perlu digeser, tanpa melihat
test set.

In [ ]:
if df is not None:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(6.4, 3))
    ax.hist(df.loc[df[LABEL_COLUMN] == 1, "probability"], bins=30, alpha=0.7, label="bully")
    ax.hist(df.loc[df[LABEL_COLUMN] == 0, "probability"], bins=30, alpha=0.7, label="non-bully")
    ax.axvline(threshold, color="#a5233a", linewidth=1, linestyle="--", label="threshold")
    ax.set_xlabel("probabilitas bully")
    ax.set_ylabel("jumlah")
    ax.legend(frameon=False)
    plt.show()
else:
    print("Dilewati: prediksi belum tersedia.")

## Ringkasan

- FN dan FP dianalisis pada validation dengan threshold yang sama seperti laporan.
- Contoh teks hanya untuk inspeksi runtime dan tidak boleh disimpan.
- Subgroup membantu melihat bias, tetapi ukuran sampel harus dihormati.
- Perbaikan berikutnya sebaiknya di data/label, bukan menambah epoch tanpa arah.